# Stage 0 — Data and problem framing

**Project:** signer-independent isolated sign language recognition from skeleton keypoints.

**What this notebook does**
1. Downloads the Slovo MediaPipe hand landmarks and the annotation file.
2. Checks the official split. **Finding on the real data: the official split is NOT signer-disjoint** (92 of 194 signers appear in both train and test), so this notebook builds its own signer-disjoint split.
3. Inspects the landmark file structure, then converts it to compact arrays (done once, cached on Drive).
4. Builds the **development subset** (a random development subset of signs, fixed seed; the extra `no_event` class is set aside) and a **signer-disjoint** train / val / test split (60 / 20 / 20, stratified by sign), while keeping the official train/test labels for comparison.
5. Computes the statistics the report needs (clip lengths, missing detections, signers per split).
6. Defines and unit-tests the two preprocessing functions used by every later stage (normalization, fixed-length resampling).

**How to run:** a **CPU runtime is enough** (Runtime → Change runtime type → None). Run the cells in order.
The JSON parsing step takes a while the first time, then the result is cached in your Drive.

**Important:** I could not download the real files while writing this notebook, so the landmark parser is written to
handle several likely layouts. Cell 7 prints the real structure. If conversion (cell 9) fails or the sanity checks
complain, paste me the output of cell 7 and I will adapt the parser.

In [ ]:
# [setup] Cell 1 — configuration
from pathlib import Path
import json, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/slovo_project')
except ImportError:                      # running outside Colab
    BASE = Path('./slovo_project')

RAW, PROC, FIG = BASE / 'raw', BASE / 'processed', BASE / 'figures'
for d in (RAW, PROC, FIG):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42                    # fixes the class subset and the signer folds
N_SUBSET_CLASSES = 300       # development subset (100 is enough to debug, 300 gives steadier val/test numbers)
N_FOLDS, TEST_FOLD, VAL_FOLD = 5, 0, 1   # signer-grouped folds -> 60% train / 20% val / 20% test
MAX_SIGNER_SHARE_OF_FOLD = 0.10          # no signer may supply more than 10% of a val/test fold (prolific signers go to train)
FIXED_T = 64                 # frames per clip after resampling (checked against the statistics below)

LANDMARKS_URL = 'https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/slovo/slovo_mediapipe.json'
SLOVO_ZIP_URL = 'https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/slovo/slovo.zip'

print('Working directory:', BASE)

In [ ]:
# Cell 2 — install small helper libraries
!pip -q install ijson remotezip tqdm

In [ ]:
# Cell 3 — download the landmark file (~1.2 GB, resumable)
LM_PATH = RAW / 'slovo_mediapipe.json'
if not LM_PATH.exists():
    !wget -c -q --show-progress -O "{LM_PATH}" "{LANDMARKS_URL}"

size_gb = LM_PATH.stat().st_size / 1e9
print(f'{LM_PATH.name}: {size_gb:.2f} GB (expected about 1.2 GB)')
assert size_gb > 0.5, 'Download looks incomplete — re-run this cell (wget resumes) or try again later.'

In [ ]:
# Cell 4 — get annotations.csv WITHOUT downloading the whole 16 GB zip
def get_annotations():
    target = RAW / 'annotations.csv'
    if target.exists():
        return target
    try:
        from remotezip import RemoteZip           # reads only the zip index + one member via HTTP range requests
        with RemoteZip(SLOVO_ZIP_URL) as z:
            matches = [n for n in z.namelist() if n.lower().endswith('annotations.csv')]
            if not matches:
                raise FileNotFoundError('annotations.csv not found inside the remote zip')
            z.extract(matches[0], RAW / '_tmp')
        (RAW / '_tmp' / matches[0]).replace(target)
        return target
    except Exception as e:
        print('Automatic download failed:', repr(e))
        print('Fallbacks:\n'
              f'  1. Upload annotations.csv by hand to {RAW}\n'
              '  2. Or use the Kaggle API (upload kaggle.json first):\n'
              f'     !kaggle datasets download -d kapitanov/slovo -f annotations.csv -p "{RAW}" --unzip\n'
              '     (check the exact file name on the dataset page)')
        raise

ann_path = get_annotations()
print('annotations:', ann_path)

In [ ]:
# Cell 5 — load the annotations and audit the official split
ann = pd.read_csv(ann_path, sep=None, engine='python')      # sniffs the delimiter (tab or comma)
expected_cols = {'attachment_id', 'user_id', 'length', 'text', 'train'}
missing_cols = expected_cols - set(ann.columns)
assert not missing_cols, f'Unexpected columns {list(ann.columns)}; missing {missing_cols}'

ann['train'] = ann['train'].astype(str).str.lower().isin(['true', '1'])
ann['attachment_id'] = ann['attachment_id'].astype(str)

# Landmark x/y are fractions of image width / height, so x and y are NOT in the same unit unless we correct for the aspect ratio.
if {'width', 'height'}.issubset(ann.columns):
    ann['aspect'] = (ann['width'] / ann['height']).astype(float)
    print('video aspect ratios (width/height):', ann['aspect'].round(2).value_counts().head(4).to_dict())
else:
    ann['aspect'] = 1.0
    print('no width/height columns: aspect ratio will not be corrected')

print('rows:', len(ann), '| classes:', ann['text'].nunique(), '| signers:', ann['user_id'].nunique())

# The extra "no_event" class (clips without a sign) is not one of the 1,000 signs: set it aside.
NO_EVENT = 'no_event'
ann_no_event = ann[ann['text'] == NO_EVENT].copy()
ann = ann[ann['text'] != NO_EVENT].copy()
print(f'set aside {len(ann_no_event)} "{NO_EVENT}" clips -> {len(ann)} sign clips, {ann["text"].nunique()} signs')

per_class = ann['text'].value_counts()
print('clips per sign: min', per_class.min(), '| max', per_class.max())

# --- audit of the OFFICIAL train/test split ---
tr_sig = set(ann.loc[ann['train'], 'user_id'])
te_sig = set(ann.loc[~ann['train'], 'user_id'])
shared = tr_sig & te_sig
test_rows = ann[~ann['train']]
unseen = ~test_rows['user_id'].isin(tr_sig)
print('\nOfficial split audit')
print(f'  train clips {int(ann.train.sum())} | test clips {int((~ann.train).sum())}')
print(f'  signers in train: {len(tr_sig)} | in test: {len(te_sig)} | in BOTH: {len(shared)}')
print(f'  official test clips from signers never seen in train: {int(unseen.sum())} '
      f'({unseen.mean():.0%} of the official test set)')
if shared:
    print('  -> the official split is NOT signer-disjoint; we will build our own signer-disjoint split (Cell 12).')
ann['official_train'] = ann['train']
ann['official_test_unseen_signer'] = (~ann['train']) & (~ann['user_id'].isin(tr_sig))

In [ ]:
# Cell 6 — streaming JSON reader (the file is too big to json.load comfortably)
import ijson
from tqdm.auto import tqdm
print('ijson backend:', ijson.backend)

def iter_clips(path):
    """Yield (clip_id, clip_object) one clip at a time, without loading the whole file."""
    with open(path, 'rb') as f:
        head = f.read(1)
        while head.isspace():
            head = f.read(1)
        f.seek(0)
        if head == b'{':                                    # {clip_id: clip, ...}
            yield from ijson.kvitems(f, '', use_float=True)
        elif head == b'[':                                  # [ {attachment_id: ..., ...}, ... ]
            for item in ijson.items(f, 'item', use_float=True):
                key = next((item[k] for k in ('attachment_id', 'id', 'video_id')
                            if isinstance(item, dict) and k in item), None)
                if key is None:
                    raise ValueError('Top level is a list but no id key was found — paste the inspect output to me.')
                yield key, item
        else:
            raise ValueError(f'Unexpected first byte {head!r}')

def describe(x, depth=0, max_depth=4):
    pad = '  ' * depth
    if depth > max_depth:
        print(pad + '...')
    elif isinstance(x, dict):
        print(f'{pad}dict({len(x)} keys): {list(x)[:6]}')
        for k in list(x)[:2]:
            print(f'{pad}  [{k!r}] ->')
            describe(x[k], depth + 2, max_depth)
    elif isinstance(x, (list, tuple)):
        print(f'{pad}list(len={len(x)})')
        if x:
            describe(x[0], depth + 1, max_depth)
    else:
        print(f'{pad}{type(x).__name__}: {x!r}'[:120])

In [ ]:
# Cell 7 — INSPECT the real structure (if anything later fails, paste this output to me)
for i, (k, v) in enumerate(iter_clips(LM_PATH)):
    print('=' * 60)
    print('clip id:', repr(k))
    describe(v)
    if i >= 1:
        break

In [ ]:
# [parse] Cell 8 — turn one clip into an array (n_frames, 2 hands, 21 joints, 3 coords)
# The parser searches each frame for "21 points" blocks, so it tolerates several layouts.
def _is_point(p):
    if isinstance(p, dict):
        return 'x' in p and 'y' in p
    return (isinstance(p, (list, tuple)) and 2 <= len(p) <= 4
            and all(isinstance(v, (int, float)) for v in p))

def _points_to_array(points):
    out = np.zeros((21, 3), np.float32)
    for i, p in enumerate(points):
        if isinstance(p, dict):
            out[i] = (p['x'], p['y'], p.get('z', 0.0))
        else:
            vals = list(p)[:3]
            out[i, :len(vals)] = vals
    return out

def find_hands(obj, label=None, found=None):
    """Recursively collect (label, array(21, 3)) for every 21-point block inside a frame."""
    if found is None:
        found = []
    if isinstance(obj, dict):
        xs = obj.get('x')
        if isinstance(xs, (list, tuple)) and len(xs) == 21 and 'y' in obj:
            z = obj.get('z', [0.0] * 21)
            found.append((label, np.stack([obj['x'], obj['y'], z], axis=1).astype(np.float32)))
            return found
        for k, v in obj.items():
            find_hands(v, str(k).lower(), found)
    elif isinstance(obj, (list, tuple)):
        if len(obj) == 21 and all(_is_point(p) for p in obj):
            found.append((label, _points_to_array(obj)))
        elif len(obj) in (42, 63) and all(isinstance(v, (int, float)) for v in obj):
            c = len(obj) // 21
            arr = np.zeros((21, 3), np.float32)
            arr[:, :c] = np.asarray(obj, np.float32).reshape(21, c)
            found.append((label, arr))
        else:
            for v in obj:
                find_hands(v, label, found)
    return found

def frame_to_slots(frame):
    """One frame -> (2, 21, 3). NaN where a hand is missing. Slot 0 = left (or first found), slot 1 = right."""
    slots = np.full((2, 21, 3), np.nan, np.float32)
    free = [0, 1]
    for label, arr in find_hands(frame):
        if np.allclose(arr, 0):                  # all-zero block = "no detection"
            continue
        if label and 'left' in label and 0 in free:
            s = 0
        elif label and 'right' in label and 1 in free:
            s = 1
        elif free:
            s = free[0]
        else:
            break
        slots[s] = arr
        free.remove(s)
    return slots

def clip_to_frames(clip):
    if isinstance(clip, list):
        return clip
    if isinstance(clip, dict):
        for key in ('frames', 'landmarks', 'hands', 'keypoints'):
            if isinstance(clip.get(key), list):
                return clip[key]
        try:                                      # dict keyed by frame index
            return [clip[k] for k in sorted(clip, key=lambda s: int(s))]
        except (ValueError, TypeError):
            pass
    raise ValueError('Unknown clip layout — paste the output of Cell 7 to me and I will adapt the parser.')

def clip_to_array(clip):
    frames = clip_to_frames(clip)
    if len(frames) == 0:
        raise ValueError('empty clip')
    return np.stack([frame_to_slots(f) for f in frames])

In [ ]:
# Cell 9 — sanity-check the parser on a few real clips before converting everything
for i, (k, v) in enumerate(iter_clips(LM_PATH)):
    arr = clip_to_array(v)
    present = ~np.isnan(arr).all(axis=(2, 3))            # (frames, 2) hand present?
    print(f'{str(k)[:12]}…  frames={len(arr)}  hands/frame={present.sum(1).mean():.2f}  '
          f'coord range x[{np.nanmin(arr[..., 0]):.2f}, {np.nanmax(arr[..., 0]):.2f}] '
          f'y[{np.nanmin(arr[..., 1]):.2f}, {np.nanmax(arr[..., 1]):.2f}]')
    if i >= 4:
        break
print('\nCheck: frames should be about 15–100, hands/frame between 0 and 2, coordinates either 0–1 (normalized) or pixels.')

In [ ]:
# Cell 10 — convert ALL clips once and cache them on Drive (first run: several minutes)
OUT_NPZ = PROC / 'landmarks_raw.npz'

def norm_id(k):
    k = str(k)
    return k.rsplit('.', 1)[0] if k.lower().endswith(('.mp4', '.avi', '.mov')) else k

if OUT_NPZ.exists():
    z = np.load(OUT_NPZ, allow_pickle=False)
    frames_raw, lengths, ids = z['frames'], z['lengths'], z['ids']
    print('Loaded cache:', OUT_NPZ.name)
else:
    wanted = set(ann['attachment_id'])
    ids_l, len_l, chunks, skipped, empty = [], [], [], 0, []
    for key, clip in tqdm(iter_clips(LM_PATH), total=len(ann)):
        cid = norm_id(key)
        if cid not in wanted:
            skipped += 1
            continue
        if len(clip_to_frames(clip)) == 0:          # some clips have no landmarks at all: record and skip them
            empty.append(cid)
            continue
        arr = clip_to_array(clip)
        ids_l.append(cid); len_l.append(len(arr)); chunks.append(arr.astype(np.float16))
    print(f'{len(empty)} clips have NO landmark frames and were skipped (ids saved to empty_clips.csv)')
    pd.Series(empty, name='attachment_id').to_csv(PROC / 'empty_clips.csv', index=False)
    frames_raw = np.concatenate(chunks)
    lengths = np.array(len_l, np.int32)
    ids = np.array(ids_l)
    np.savez_compressed(OUT_NPZ, frames=frames_raw, lengths=lengths, ids=ids)
    print(f'Saved {OUT_NPZ.name}; clips not in annotations (skipped): {skipped}')

offsets = np.concatenate([[0], np.cumsum(lengths)])
id2idx = {c: i for i, c in enumerate(ids)}
print('clips:', len(ids), '| total frames:', len(frames_raw), '| array shape:', frames_raw.shape, frames_raw.dtype)

def get_clip(cid):
    """Return one RAW clip as float32 array (n_frames, 2, 21, 3). (Cell 11b redefines this with stable hand slots.)"""
    i = id2idx[cid]
    return frames_raw[offsets[i]:offsets[i + 1]].astype(np.float32)

In [ ]:
# Cell 11 — match landmarks to annotations
match_rate = ann['attachment_id'].isin(list(id2idx)).mean()
print(f'{match_rate:.1%} of annotated clips have landmarks')
assert match_rate > 0.95, 'Too few clip ids match — the id format differs; paste Cell 7 output to me.'
assert (lengths > 0).all()

n_before = len(ann)
ann = ann[ann['attachment_id'].isin(list(id2idx))].copy()
print(f'dropped {n_before - len(ann)} clips without landmarks | signs: {ann["text"].nunique()} | '
      f'min clips per sign now: {ann["text"].value_counts().min()}')
ann['n_frames'] = ann['attachment_id'].map(lambda c: int(lengths[id2idx[c]]))

hands_per_frame = (~np.isnan(frames_raw).all(axis=(2, 3))).sum(1)             # 0, 1 or 2 hands
print('frames with 0 / 1 / 2 hands:', [f'{(hands_per_frame == k).mean():.1%}' for k in (0, 1, 2)])

# The file lists no frame without a detection, so "frames with 0 hands" is 0% by construction. A real gap shows up
# as: video length (annotations) - number of landmark frames.
if 'length' in ann.columns:
    diff = ann['length'] - ann['n_frames']
    ann['missing_rate'] = diff.clip(lower=0) / ann['length'].clip(lower=1)
    print(f'annotation length == landmark frames: {(diff == 0).mean():.1%} of clips | '
          f'more video frames than landmarks: {(diff > 0).mean():.1%} | fewer: {(diff < 0).mean():.1%}')
    print(f'difference (video - landmarks): median {diff.median():.0f} | mean {diff.mean():.2f} | '
          f'95th pct {diff.quantile(.95):.0f} | max {diff.max()}')
    print(f'estimated share of video frames absent from the landmark file: mean {ann.missing_rate.mean():.1%}')
else:
    ann['missing_rate'] = 0.0
ann['low_coverage'] = ann['missing_rate'] > 0.5      # clips missing more than half of their frames

# --- are "hand 1" / "hand 2" a stable left/right ordering? compare the wrist x of slot 0 and slot 1 in two-hand frames
def slot_order_stats(fr, offs):
    wx0 = fr[:, 0, 0, 0].astype(np.float32)
    wx1 = fr[:, 1, 0, 0].astype(np.float32)
    both = ~np.isnan(wx0) & ~np.isnan(wx1)
    if both.sum() == 0:
        return None
    slot0_left = both & (wx0 < wx1)
    n_both = np.add.reduceat(both.astype(np.int64), offs[:-1])
    frac = np.add.reduceat(slot0_left.astype(np.int64), offs[:-1]) / np.maximum(n_both, 1)
    ok = n_both >= 5
    unstable = ok & (frac > 0.1) & (frac < 0.9)
    return slot0_left.sum() / both.sum(), int(unstable.sum()), int(ok.sum())

def print_slot_stats(fr, offs, label):
    st = slot_order_stats(fr, offs)
    if st:
        print(f'[{label}] two-hand frames where slot 0 is left of slot 1: {st[0]:.1%} | '
              f'clips whose slot order flips along the clip: {st[1]} of {st[2]} ({st[1] / max(st[2], 1):.1%})')

print_slot_stats(frames_raw, offsets, 'raw detection order')

In [ ]:
# [track] Cell 11b — give each hand a stable slot (the file only says "hand 1 / hand 2" = detection order)
def _wrist_dist(a, b):
    return 0.0 if b is None else float(np.hypot(a[0] - b[0], a[1] - b[1]))

def track_slots(clip):
    """clip (L, 2, 21, 3) with slot = detection order -> same shape, but each physical hand keeps ONE slot.
    Hands are matched between consecutive frames by wrist position."""
    out = np.full_like(clip, np.nan)
    last = [None, None]                                   # last wrist (x, y) seen in each slot
    for t in range(len(clip)):
        dets = [clip[t, s] for s in (0, 1) if not np.isnan(clip[t, s]).all()]
        if not dets:
            continue
        w = [d[0, :2] for d in dets]
        if len(dets) == 2:
            if last[0] is None and last[1] is None:
                slots = (0, 1) if w[0][0] <= w[1][0] else (1, 0)          # first time: leftmost hand -> slot 0
            else:
                keep = _wrist_dist(w[0], last[0]) + _wrist_dist(w[1], last[1])
                swap = _wrist_dist(w[0], last[1]) + _wrist_dist(w[1], last[0])
                slots = (0, 1) if keep <= swap else (1, 0)
        elif last[0] is None and last[1] is None:
            slots = (0 if w[0][0] < 0.5 else 1,)
        else:
            d0 = _wrist_dist(w[0], last[0]) if last[0] is not None else np.inf
            d1 = _wrist_dist(w[0], last[1]) if last[1] is not None else np.inf
            slots = (0 if d0 <= d1 else 1,)
        for det, slot, wr in zip(dets, slots, w):
            out[t, slot] = det
            last[slot] = wr
    return out

def jump_rate(fr, offs, thr=0.2):
    """Share of consecutive frames (same clip, same slot) where the wrist moves more than thr (fraction of the image)."""
    wr = fr[:, :, 0, :2].astype(np.float32)
    d = np.linalg.norm(wr[1:] - wr[:-1], axis=-1)
    same_clip = np.ones(len(fr) - 1, bool)
    same_clip[offs[1:-1] - 1] = False
    valid = same_clip[:, None] & ~np.isnan(d)
    return float(((d > thr) & valid).sum() / max(valid.sum(), 1))

# --- unit check: two smoothly moving hands whose slots are randomly swapped must be un-swapped ---
_rng = np.random.default_rng(0)
_T = 40
_a = np.zeros((_T, 21, 3), np.float32); _b = np.zeros((_T, 21, 3), np.float32)
_a[:] = (0.25 + 0.004 * np.arange(_T))[:, None, None]; _b[:] = (0.75 - 0.004 * np.arange(_T))[:, None, None]
_mixed = np.stack([_a, _b], axis=1).copy()
for _t in range(_T):
    if _rng.random() < 0.5:
        _mixed[_t] = _mixed[_t][::-1]
_fixed = track_slots(_mixed)
_o = np.array([0, _T])
assert jump_rate(_mixed, _o) > 0.2 and jump_rate(_fixed, _o) == 0.0, 'tracking unit check failed'
assert np.allclose(_fixed[:, 0, 0, 0], _a[:, 0, 0]) and np.allclose(_fixed[:, 1, 0, 0], _b[:, 0, 0]), 'identity not preserved'
print('OK: tracking unit check passed')

# --- apply to all clips (cached) ---
TRACKED = PROC / 'frames_tracked_v2.npy'     # new name: ignores any stale cache from an earlier version
if TRACKED.exists():
    frames_tracked = np.load(TRACKED)
else:
    frames_tracked = np.empty_like(frames_raw)
    for i in tqdm(range(len(ids))):
        a, b = offsets[i], offsets[i + 1]
        frames_tracked[a:b] = track_slots(frames_raw[a:b].astype(np.float32)).astype(np.float16)
    np.save(TRACKED, frames_tracked)

jr_before, jr_after = jump_rate(frames_raw, offsets), jump_rate(frames_tracked, offsets)
print(f'wrist jumps > 0.2 between consecutive frames: before tracking {jr_before:.2%} | after {jr_after:.2%}')
print_slot_stats(frames_tracked, offsets, 'after tracking (remaining flips = hands that really cross)')

def get_clip(cid, tracked=True):
    """One clip as float32 (n_frames, 2, 21, 3). tracked=True: each hand keeps one slot (this cell)."""
    i = id2idx[cid]
    src = frames_tracked if tracked else frames_raw
    return src[offsets[i]:offsets[i + 1]].astype(np.float32)

In [ ]:
# [splits] Cell 12 — signer-disjoint split (prolific signers always in train) + the development subset
for _name in ('N_FOLDS', 'TEST_FOLD', 'VAL_FOLD', 'MAX_SIGNER_SHARE_OF_FOLD'):
    assert _name in globals(), 'Cell 1 is outdated in this session: restart the runtime (Runtime > Restart session) and run the cells from Cell 1.'


def _fold_assignment(df, seed, n_folds, noise=0.35):
    """Balanced signer folds: signers (largest first, sizes slightly perturbed by the seed) are put one by one into the
    currently smallest fold. Every signer lands in exactly ONE fold and the folds get (almost) equal numbers of clips."""
    r = np.random.default_rng(seed)
    counts = df['user_id'].value_counts()
    order = (counts * np.exp(r.normal(0, noise, len(counts)))).sort_values(ascending=False).index
    load, fold_of = np.zeros(n_folds), {}
    for u in order:
        k = int(np.argmin(load))
        fold_of[u] = k
        load[k] += counts[u]
    return df['user_id'].map(fold_of).to_numpy()

def find_prolific_signers(df, n_folds, max_share, min_pool_share=0.30):
    """Signers who would supply more than max_share of a val/test fold. They are removed from the fold pool
    (and always trained on); the threshold is recomputed after each removal. Safeguard: stop before the pool
    would keep less than 30% of the clips or fewer than 10 signers per fold."""
    counts = df['user_id'].value_counts()
    total, heavy = counts.sum(), []
    while True:
        fold_size = counts.sum() / n_folds
        big = counts[counts > max_share * fold_size]
        if big.empty:
            return heavy
        rest = counts.drop(big.index)
        if rest.sum() < min_pool_share * total or len(rest) < 10 * n_folds:
            print(f'NOTE: stopped removing prolific signers ({len(heavy)} so far) to keep enough signers for the folds; '
                  'the largest signer share in a fold may stay above the target.')
            return heavy
        heavy += big.index.tolist()
        counts = rest

def largest_share(df):
    return float(df['user_id'].value_counts().iloc[0] / len(df)) if len(df) else 0.0

def make_splits(ann, seed=SEED, n_classes=N_SUBSET_CLASSES, n_folds=N_FOLDS, test_fold=TEST_FOLD,
                val_fold=VAL_FOLD, max_share=MAX_SIGNER_SHARE_OF_FOLD, n_tries=60):
    """1. Development subset = seeded random sample of signs.
    2. Prolific signers (a few signers recorded most clips) are always in TRAIN, otherwise a single person could be
       most of the validation or test set.
    3. All other signers go entirely to ONE of n_folds folds (equal clip counts), computed on ALL 1,000 signs, so the
       subset inherits the same signer assignment. Several assignments are tried; the one with the smallest signer
       dominance, then the most balanced folds, then the best sign coverage, is kept (only clip COUNTS are used,
       never model results)."""
    rng = np.random.default_rng(seed)
    classes = sorted(ann['text'].unique())
    subset = sorted(rng.choice(classes, size=n_classes, replace=False).tolist())
    out = ann.reset_index(drop=True).copy()
    out['in_subset'] = out['text'].isin(subset)

    heavy = find_prolific_signers(out, n_folds, max_share)
    out['prolific_signer'] = out['user_id'].isin(heavy)
    pool = out[~out['prolific_signer']]

    best = None
    sub_pool = pool['in_subset'].to_numpy()
    for k in range(n_tries):
        fold = _fold_assignment(pool, seed + k, n_folds)
        dom = max(largest_share(pool[fold == f]) for f in range(n_folds))
        sizes = np.bincount(fold, minlength=n_folds) / len(fold)
        cov = pd.crosstab(pool.loc[sub_pool, 'text'], fold[sub_pool]).reindex(subset, fill_value=0)
        zeros = int((cov[[val_fold, test_fold]].to_numpy() == 0).sum())
        imbalance = float(np.abs(sizes - 1 / n_folds).sum())
        score = (round(dom / 0.05), round(imbalance / 0.05), zeros)          # coarse buckets, then coverage
        if best is None or score < best[0]:
            best = (score, k, fold, dom)
    print(f'{len(heavy)} prolific signers always in train ({int(out["prolific_signer"].sum())} clips, '
          f'{out["prolific_signer"].mean():.0%} of all clips)')
    print(f'fold assignment: try #{best[1]} of {n_tries} | largest signer share in any fold {best[3]:.0%} | '
          f'(sign, fold) cells without a clip in val/test for the development subset: {best[0][2]}')

    out['fold'] = -1
    out.loc[pool.index, 'fold'] = best[2]
    out['split'] = np.select([out['fold'] == test_fold, out['fold'] == val_fold], ['test', 'val'], default='train')
    out['label_id_full'] = out['text'].map({c: i for i, c in enumerate(classes)})
    out['label_id_subset'] = out['text'].map({c: i for i, c in enumerate(subset)})
    return out, subset

def check_disjoint(df):
    sig = {s: set(df.loc[df['split'] == s, 'user_id']) for s in ('train', 'val', 'test')}
    for a, b in (('train', 'val'), ('train', 'test'), ('val', 'test')):
        assert not (sig[a] & sig[b]), f'signer leakage between {a} and {b}'

ann, subset_classes = make_splits(ann)
check_disjoint(ann)
print('OK: our train / val / test split shares no signers')

def split_table(df):
    rows = []
    for s_ in ('train', 'val', 'test'):
        d = df[df['split'] == s_]
        rows.append({'split': s_, 'clips': len(d), 'signers': d['user_id'].nunique(),
                     'largest signer share': f'{largest_share(d):.0%}'})
    return pd.DataFrame(rows).set_index('split')

sub = ann[ann['in_subset']]
print('\nAll 1,000 signs');  print(split_table(ann))
print(f'\nDevelopment subset ({N_SUBSET_CLASSES} signs)');  print(split_table(sub))
for s_ in ('val', 'test'):
    pc = sub[sub['split'] == s_]['text'].value_counts().reindex(subset_classes, fill_value=0)
    print(f'Subset {s_}: clips per sign  min {pc.min()}  median {int(pc.median())}  max {pc.max()}  '
          f'| signs with no clip: {(pc == 0).sum()}')
print('\nMost prolific signers (clips over all 1,000 signs):')
print(ann['user_id'].value_counts().head(8).rename('clips').to_frame().assign(split=lambda d: [ann.loc[ann.user_id == u, 'split'].iloc[0] for u in d.index]))

In [ ]:
# [prep] Cell 13 — preprocessing functions used by every later stage (+ unit checks)
WRIST, MIDDLE_MCP = 0, 9

def normalize_hands(x, aspect=None, eps=1e-6):
    """x: (..., 21, 3). Centre on the wrist, scale by the wrist->middle-finger-base distance (xy).
    Makes features invariant to hand position and to distance from the camera. NaN stays NaN.
    aspect = video width/height (scalar): converts x from fraction-of-width to fraction-of-height so that x and y
    share one unit before the scale is computed."""
    x = x.astype(np.float32).copy()
    if aspect is not None:
        x[..., 0] *= aspect
    centred = x - x[..., WRIST:WRIST + 1, :]
    scale = np.linalg.norm(centred[..., MIDDLE_MCP, :2], axis=-1)
    scale = np.where(scale < eps, np.nan, scale)
    return centred / scale[..., None, None]

def resample_fixed_length(clip, T=FIXED_T):
    """clip: (L, 2, 21, 3) -> (T, 2, 21, 3) by nearest-index resampling, plus a (T,) mask of frames with a hand."""
    L = len(clip)
    idx = np.linspace(0, L - 1, T).round().astype(int)
    out = clip[idx]
    valid = ~np.isnan(out).all(axis=(1, 2, 3))
    return out, valid

# --- unit checks on synthetic data ---
rng = np.random.default_rng(0)
h = rng.normal(size=(5, 2, 21, 3)).astype(np.float32)
h_moved = h * 3.0 + np.array([10.0, -4.0, 0.0], np.float32)      # same hand, bigger and shifted
assert np.allclose(normalize_hands(h), normalize_hands(h_moved), atol=1e-4), 'not invariant to shift/scale'
h_nan = h.copy(); h_nan[2, 1] = np.nan
assert np.allclose(normalize_hands(h, aspect=2.0), normalize_hands(np.concatenate([h[..., :1] * 2.0, h[..., 1:]], -1)), atol=1e-5)
assert np.isnan(normalize_hands(h_nan)[2, 1]).all() and not np.isnan(normalize_hands(h_nan)[2, 0]).any()
short, long_ = rng.normal(size=(20, 2, 21, 3)), rng.normal(size=(120, 2, 21, 3))
assert resample_fixed_length(short)[0].shape == (FIXED_T, 2, 21, 3)
assert resample_fixed_length(long_)[0].shape == (FIXED_T, 2, 21, 3)
print('OK: preprocessing unit checks passed')

In [ ]:
# Cell 14 — statistics for the report
L = ann['n_frames'].to_numpy()
pct = {p: float(np.percentile(L, p)) for p in (5, 25, 50, 75, 95, 99)}
print('clip length (frames) percentiles:', pct, '| mean', round(float(L.mean()), 1))
print(f'-> FIXED_T = {FIXED_T} covers {np.mean(L <= FIXED_T):.0%} of clips without needing to subsample')

for thr in (0.3, 0.5):
    print(f'clips missing more than {thr:.0%} of their frames: {int((ann.missing_rate > thr).sum())}')

worst = (ann.groupby('user_id')['missing_rate'].mean().sort_values(ascending=False).head(5))
print('\nSigners with the highest missing-detection rate:')
print(worst.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].hist(L, bins=40, color='#4C78A8'); axes[0].axvline(FIXED_T, color='k', ls='--', label=f'T={FIXED_T}')
axes[0].set_title('Clip length (frames)'); axes[0].legend()
for s, c in (('train', '#4C78A8'), ('val', '#F58518'), ('test', '#54A24B')):
    v = np.sort(ann[ann['split'] == s].groupby('user_id').size().to_numpy())[::-1]
    axes[1].plot(np.arange(1, len(v) + 1), v, marker='.', ms=3, label=f'{s} ({len(v)} signers)', color=c)
axes[1].set_yscale('log'); axes[1].set_xlabel('signer rank'); axes[1].set_title('Clips per signer (sorted, log scale)'); axes[1].legend()
axes[2].hist(ann['missing_rate'], bins=30, color='#B279A2')
axes[2].set_title('Estimated share of video frames absent from landmarks')
plt.tight_layout(); plt.savefig(FIG / 'stage0_overview.png', dpi=150); plt.show()

In [ ]:
# Cell 15 — look at a few signs (frames of the normalized hands)
HAND_CONNECTIONS = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11),
                    (11, 12), (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]

def plot_clip_row(axes_row, cid, title):
    clip = get_clip(cid)
    clip[..., 0] *= float(ann.loc[ann['attachment_id'] == cid, 'aspect'].iloc[0])     # x: fraction of width -> same unit as y
    idx = np.linspace(0, len(clip) - 1, len(axes_row)).round().astype(int)
    for ax, t in zip(axes_row, idx):
        for slot, color in ((0, '#1f77b4'), (1, '#ff7f0e')):
            pts = clip[t, slot]
            if np.isnan(pts).all():
                continue
            for a, b in HAND_CONNECTIONS:
                ax.plot([pts[a, 0], pts[b, 0]], [pts[a, 1], pts[b, 1]], color=color, lw=1)
            ax.scatter(pts[:, 0], pts[:, 1], s=6, color=color)
        ax.set_aspect('equal'); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'frame {t}', fontsize=8)
    axes_row[0].set_ylabel(title, fontsize=9)

rng = np.random.default_rng(SEED)
show = rng.choice(subset_classes, size=3, replace=False)
fig, axes = plt.subplots(len(show), 5, figsize=(12, 7))
for r, cls in enumerate(show):
    cid = sub[(sub['text'] == cls) & (sub['split'] == 'train')]['attachment_id'].iloc[0]
    plot_clip_row(axes[r], cid, str(cls))
plt.suptitle('Landmarks of three development-subset signs after hand tracking, aspect-corrected (blue = slot 0, orange = slot 1)')
plt.tight_layout(); plt.savefig(FIG / 'stage0_examples.png', dpi=150); plt.show()

In [ ]:
# Cell 16 — save the split files and a summary for the report
if 'low_coverage' not in ann.columns:
    ann['low_coverage'] = ann['missing_rate'] > 0.5
cols = ['attachment_id', 'user_id', 'text', 'label_id_full', 'label_id_subset', 'in_subset', 'split', 'fold',
        'prolific_signer', 'official_train', 'official_test_unseen_signer', 'aspect', 'n_frames', 'missing_rate', 'low_coverage']
ann[cols].to_csv(PROC / 'splits.csv', index=False)
pd.Series(subset_classes, name='text').to_csv(PROC / 'subset_classes.csv', index_label='label_id_subset')
ann_no_event[['attachment_id', 'user_id']].to_csv(PROC / 'no_event_ids.csv', index=False)
ann.groupby('user_id').agg(clips=('attachment_id', 'size'), prolific=('prolific_signer', 'first'), split=('split', 'first')) \
   .sort_values('clips', ascending=False).to_csv(PROC / 'signers.csv')

summary = {
    'seed': SEED, 'n_subset_classes': N_SUBSET_CLASSES, 'folds': {'n': N_FOLDS, 'test': TEST_FOLD, 'val': VAL_FOLD},
    'n_sign_clips': int(len(ann)), 'n_no_event_clips': int(len(ann_no_event)), 'n_signers': int(ann['user_id'].nunique()),
    'official_split': {
        'train_clips': int(ann['official_train'].sum()), 'test_clips': int((~ann['official_train']).sum()),
        'signers_in_train': len(tr_sig), 'signers_in_test': len(te_sig), 'signers_in_both': len(shared),
        'test_clips_from_unseen_signers': int(ann['official_test_unseen_signer'].sum())},
    'prolific_signers': {'n': int(ann.loc[ann['prolific_signer'], 'user_id'].nunique()), 'clips': int(ann['prolific_signer'].sum())},
    'all_signs': {s: {'clips': int((ann['split'] == s).sum()), 'signers': int(ann.loc[ann['split'] == s, 'user_id'].nunique()),
                      'largest_signer_share': largest_share(ann[ann['split'] == s])} for s in ('train', 'val', 'test')},
    'subset': {s: {'clips': int((sub['split'] == s).sum()), 'signers': int(sub.loc[sub['split'] == s, 'user_id'].nunique()),
                     'largest_signer_share': largest_share(sub[sub['split'] == s])} for s in ('train', 'val', 'test')},
    'clips_missing_over_50pct_frames': int(ann['low_coverage'].sum()),
    'clip_length_percentiles': pct, 'clip_length_mean': float(L.mean()),
    'mean_missing_rate': float(ann['missing_rate'].mean()),
    'frames_with_0_1_2_hands': [float((hands_per_frame == k).mean()) for k in (0, 1, 2)],
    'fixed_T': FIXED_T,
    'wrist_jump_rate': {'before_tracking': jr_before, 'after_tracking': jr_after},
}
(PROC / 'stage0_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print('\nSaved to', PROC)

## Stage 0 checklist

- [ ] Cell 5 printed the official-split audit (expected: 92 signers in both train and test)
- [ ] Cell 9 printed sensible frame counts, hands per frame and coordinate ranges
- [ ] Cell 11 matched more than 95% of clips
- [ ] Cell 12 printed `OK: our train / val / test split shares no signers`
- [ ] Cell 13 unit checks passed
- [ ] `splits.csv`, `subset_classes.csv`, `landmarks_raw.npz` and `stage0_summary.json` are in your Drive

**Send back:** the output of Cell 7 (structure), the printed summary of Cell 16 and the two figures. With these I will
fill the results part of the report and write Stage 1 (classical baseline) on top of `splits.csv` and `landmarks_raw.npz`.